# Week 3-1 · 역할별 state ownership 분리하기

## 시나리오
triage, investigator, commander 역할이 하나의 incident state를 공유하되 각자 허용된 필드만 갱신하도록 만듭니다.

## 학습 목표
- 공유 `IncidentState`의 필드를 역할별로 구분한다.
- 각 node가 자기 소유 필드만 반환하도록 작성한다.
- audit trace로 실행 순서와 변경 필드를 확인한다.

## 직접 조립
완성된 `weekX.app` 함수를 가져오지 않습니다. 아래 코드에서 작은 fixture와 핵심 객체·함수·연결을 직접 만듭니다.

### 1단계 · state와 ownership 선언

In [ ]:
# 실행 순서: 1단계 · state와 ownership 선언에서 IncidentState, triage_node, investigator_node을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 1단계 · state와 ownership 선언.
from typing import TypedDict

# 역할별 산출물과 비실행 필드를 함께 두어 write ownership을 검사합니다.
class IncidentState(TypedDict):
    service: str
    summary: str
    severity: str
    objective: str
    findings: list[str]
    proposals: list[str]
    agents_run: list[str]
    executed_commands: list[str]

OWNERSHIP = {
    "triage": {"objective", "agents_run"},
    "investigator": {"findings", "agents_run"},
    "commander": {"proposals", "agents_run"},
}

# triage 역할이 objective와 자신의 trace만 갱신합니다.
def triage_node(state: IncidentState) -> dict:
    return {"objective": f'{state["service"]} 영향 범위 확인', "agents_run": state["agents_run"] + ["triage"]}

# investigator 역할이 관찰 findings와 자신의 trace만 갱신합니다.
def investigator_node(state: IncidentState) -> dict:
    return {"findings": ["최근 배포와 오류율을 비교"], "agents_run": state["agents_run"] + ["investigator"]}

# commander 역할이 실행하지 않을 read-only proposal만 작성합니다.
def commander_node(state: IncidentState) -> dict:
    return {"proposals": ["kubectl rollout status deployment/checkout"], "agents_run": state["agents_run"] + ["commander"]}

### 2단계 · 소유권 검사와 순차 적용

In [ ]:
# 실행 순서: 2단계 · 소유권 검사와 순차 적용에서 apply_owned_update을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 2단계 · 소유권 검사와 순차 적용.
# 역할의 허용 필드 밖 update를 merge 전에 거부합니다.
def apply_owned_update(state: IncidentState, role: str, update: dict) -> IncidentState:
    unexpected = set(update) - OWNERSHIP[role]
    if unexpected:
        raise ValueError(f"{role} cannot write: {sorted(unexpected)}")
    return {**state, **update}

practice_state: IncidentState = {
    "service": "checkout", "summary": "배포 후 오류", "severity": "SEV1",
    "objective": "", "findings": [], "proposals": [], "agents_run": [], "executed_commands": [],
}
for role, node in [("triage", triage_node), ("investigator", investigator_node), ("commander", commander_node)]:
    practice_state = apply_owned_update(practice_state, role, node(practice_state))
practice_state

### 3단계 · 경계 위반과 비실행 확인

In [ ]:
# 실행 순서: 3단계 · 경계 위반과 비실행 확인에서 fixture와 assertion을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 3단계 · 경계 위반과 비실행 확인.
try:
    apply_owned_update(practice_state, "commander", {"executed_commands": ["danger"]})
except ValueError as error:
    practice_ownership_error = str(error)

assert practice_state["agents_run"] == ["triage", "investigator", "commander"]
assert practice_state["executed_commands"] == []
assert "cannot write" in practice_ownership_error
{"state": practice_state, "rejected": practice_ownership_error}

## 중간 결과
각 코드 셀의 출력에서 입력이 어떤 상태로 변했는지 확인합니다. 마지막 `assert`는 눈으로 본 결과를 실행 가능한 계약으로 고정합니다.

## 실패 경계
commander는 `executed_commands`를 쓸 수 없습니다. 역할 밖 필드 변경은 검증에서 실패해야 합니다.

## 실제 app 연결
Week 3 app은 이 소유권 경계에 LangGraph와 live/fixture service를 결합합니다. 멀티에이전트의 핵심은 역할 이름이 아니라 state write 책임의 분리입니다.

### 확장 과제
fixture의 문장이나 임계값을 하나 바꾸고, 어느 중간 결과와 assertion이 달라지는지 기록하세요.

## 다음 Notebook 연결
다음 `02_bounded_revision_loop.ipynb`에서는 역할별 node를 조건부 재검토 loop로 연결하고 종료 상한을 둡니다.